In [1]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import jax
print("JAX device:", jax.devices())
jax.config.update('jax_disable_jit', False) # Turn off JIT because of an issue in shortwave_radiation.py:169
jax.config.update("jax_debug_infs", True) # doesn't add any time since the saved time is otherwise spent getting the nodal quantities
jax.config.update("jax_debug_nans", False) # some physics fields might be nan

JAX device: [CpuDevice(id=0)]


In [2]:
import sys
from pathlib import Path

paths_check = [
    (Path(os.path.abspath(".")) / ".." / ".." / "jax-gcm").resolve(),
    (Path(os.path.abspath(".")) / "..").resolve(),
]

for module_path in paths_check:
    module_path = str(module_path)
    if module_path in sys.path:
        print("Path exist: ", module_path)
    else:
        print("Add Path: ", module_path)
        sys.path.append(module_path)


Add Path:  /home/t2hsu/projects/jax-gcm
Add Path:  /home/t2hsu/projects/jax-esm


In [3]:
import numpy as np
#import jcm
#from jcm.model import Model, get_coords
#from jcm.boundaries import initialize_boundaries

from jax_esm.Master import Master
from jax_esm.components.base import ComponentConfig
from jax_esm.components.Speedy import Speedy
from jax_esm.components.SlabOceanModel import SlabOceanModel
from jax_esm.components.FluxModel import FluxModel

Path exist:  /home/t2hsu/projects/jax-gcm
Path exist:  /home/t2hsu/projects/jax-esm


# Run the speedy model

In [4]:
# Master Config
config_master = dict(
    time_step = 3600.0, # sec
)

# Atmosphere model
config_atm = ComponentConfig(
    name = "atm",
    timestep = 600.0, # seconds
    grid = None,
    params = None,
)

config_speedy = dict(
    save_interval=5,
    total_time=60,
)
model_atm = Speedy(
    config = config_atm,
    config_speedy = config_speedy,
)

# Ocean model
config_ocn = ComponentConfig(
    name = "ocn",
    timestep = 600.0, # seconds
    grid = None,
    params = None,
)
model_ocn = SlabOceanModel(config_ocn)

# Flux model
config_flx = ComponentConfig(
    name = "flx",
    timestep = 600.0, # seconds
    grid = None,
    params = None,
)
model_flx = FluxModel(config_flx)

In [7]:
master = Master(
    config = config_master,
    components = dict(
        flx = model_flx,
        atm = model_atm,
        ocn = model_ocn,
    ), 
)

master.checkPlan()
master.printPlan()

import time

t0 = time.time()

state = model_atm.model.get_initial_state()
#final_state, predictions = model.unroll(state)

t1 = time.time()

total_time = t1-t0
print(f"Total time: {total_time:.1f} s.")

Print execution plan:
[ 1] : flx 
[ 2] : atm 
[ 3] : ocn 
Total time: 0.0 s.


In [8]:
time_length = 86400.0 * 10
timesteps = int(time_length / master.config["time_step"])

for timestep in range(timesteps):
    print(f"Timestep: {timestep:d}")
    master.run()

Timestep: 0


AttributeError: 'SpeedyState' object has no attribute 'T'

In [ ]:
pred_ds = model.predictions_to_xarray(predictions)

In [ ]:
print(f"dataset size: {pred_ds.nbytes/1e6:.1f}MB")

In [ ]:
pred_ds

In [ ]:
pred_ds['normalized_surface_pressure'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

In [ ]:
pred_ds['u_wind'].mean('lon').plot(x='lat', y='level', col='time', col_wrap=3, aspect=6, yincrease=False)
pred_ds['u_wind'].isel(level=-1).plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

In [ ]:
pred_ds['v_wind'].mean('lon').plot(x='lat', y='level', col='time', col_wrap=3, aspect=6, yincrease=False)
pred_ds['v_wind'].isel(level=-1).plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

In [ ]:
pred_ds['temperature'].mean('lon').plot(x='lat', y='level', col='time', col_wrap=2, aspect=6, yincrease=False)

In [ ]:
pred_ds['specific_humidity'].mean('lon').plot(x='lat', y='level', col='time', col_wrap=3, aspect=6, yincrease=False)
pred_ds['specific_humidity'].isel(level=3).plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

In [ ]:
pred_ds['shortwave_rad.cloudc'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)
pred_ds['shortwave_rad.qcloud'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)
pred_ds['shortwave_rad.icltop'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)
pred_ds['shortwave_rad.cloudstr'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)